In [ ]:
#pip install docling ## can't do that here
%pip install langchain-text-splitters

In [ ]:
%%sql -r dataframe_1
-- using cortex to extract text from pdf
USE ROLE SYSADMIN;
USE DATABASE PHARMA_COPILOT;
USE SCHEMA PHARMA_DATA;

SELECT 
    RELATIVE_PATH as file_name,
    -- in the past it would be PARSE_DOCUMENT
    SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
        TO_FILE('@pharma_pdf_stage', RELATIVE_PATH),
        OBJECT_CONSTRUCT('mode', 'LAYOUT')):content::STRING AS parsed_text
    -- :content traverses the returned JSON, ::STRING is just a cast, could also use CAST()
FROM DIRECTORY('@pharma_pdf_stage');

In [ ]:
import snowflake.snowpark as snowpark
import pandas as pd
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

In [ ]:
session = snowpark.context.get_active_session()
raw_docs_df = dataframe_1.to_pandas() # grab the output of previous cell

headers_to_split_on = [
    ("#", "Header_1"),
    ("##", "Header_2"),
    ("###", "Header_3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
char_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

chunk_data = []

for index, row in raw_docs_df.iterrows():
    text = row['PARSED_TEXT']
    if text:
        md_header_splits = markdown_splitter.split_text(text)
        final_splits = char_splitter.split_documents(md_header_splits)
        
        for i, doc in enumerate(final_splits):
            header_context = " | ".join([f"{k}: {v}" for k, v in doc.metadata.items()])
            header_and_content = f"[{header_context}]\n{doc.page_content}" if header_context else doc.page_content
            
            chunk_data.append({
                "FILE_NAME": row['FILE_NAME'],
                "CHUNK_ID": i + 1,
                "CHUNK_TEXT": header_and_content
            })

if chunk_data:
    final_df = session.create_dataframe(pd.DataFrame(chunk_data))
    final_df.write.mode("overwrite").save_as_table("DOC_CHUNKS")
else:
    raise RuntimeError("No data found in stage")